# NB11 — Ridge Regression (L2 Regularisation)

> **StatQuest: "Ridge adds a penalty that shrinks large coefficients — preventing them from going crazy just to fit the noise."**

---

## The main ideas:

1. OLS minimises SSR. Ridge minimises: **SSR + lambda * sum(b_j^2)**
2. The lambda * sum(b_j^2) term is the "L2 penalty"
3. Large lambda -> coefficients are forced toward zero -> less overfitting
4. lambda = 0 -> pure OLS. lambda -> infinity -> all slopes = 0
5. Ridge NEVER sets a coefficient exactly to zero (it shrinks, but doesn't eliminate)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    n = len(steps)
    default_colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A',
                      '#00695C','#AD1457','#37474F','#4E342E',
                      '#0277BD','#558B2F','#C62828','#F57F17']
    colors = (colors or default_colors)[:n]
    notes  = notes or ['']*n
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n*3.1); ax.set_ylim(-1.2, 2.4); ax.axis('off')
    bw, bh = 2.6, 1.3
    for i,(step,color,note) in enumerate(zip(steps,colors,notes)):
        x = i*3.1
        box = FancyBboxPatch((x,0.2),bw,bh,boxstyle="round,pad=0.12",
                             facecolor=color,edgecolor='white',linewidth=1.5,alpha=0.90)
        ax.add_patch(box)
        ax.text(x+bw/2,0.2+bh/2,step,ha='center',va='center',fontsize=8.5,
                color='white',fontweight='bold',multialignment='center')
        if note:
            ax.text(x+bw/2,0.02,note,ha='center',va='top',fontsize=7,
                    color='#555',style='italic')
        if i < n-1:
            ax.annotate('',xy=(x+bw+0.38,0.2+bh/2),xytext=(x+bw+0.08,0.2+bh/2),
                       arrowprops=dict(arrowstyle='->',color='#444',lw=2.2))
    ax.set_title(title,fontsize=11,fontweight='bold',pad=6,color='#222')
    plt.tight_layout(pad=0.4); plt.show()

flow_diagram(
    steps=[
        'OLS objective:\nminimise SSR\n(no constraint)',
        'Ridge adds\nL2 penalty:\nSSR + lambda*sum(b^2)',
        'Larger lambda\n-> smaller\ncoefficients',
        'Tune lambda\nwith\n5-fold CV',
        'Closed form:\nb = (XtX + lI)^-1 Xty',
        'Stabilises\nmulticollinear\nfeatures',
        'No variable\nselection\n(all kept)',
    ],
    title='NB11 Conceptual Flow: Ridge Regression (L2)',
    colors=['#37474F','#1565C0','#2E7D32','#E65100','#6A1B9A','#C62828','#00695C'],
    figsize=(16, 2.8),
)


## Why L2 stabilises multicollinear models

The OLS normal equations: `(XtX) b = Xty`

When predictors are correlated, XtX is near-singular (almost uninvertible).
Ridge adds `lambda*I` to the diagonal:

```
b_ridge = (XtX + lambda*I)^-1  Xty
```

Adding lambda to the diagonal makes the matrix **always invertible** — no matter how bad the multicollinearity.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, RidgeCV, LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
n = 50
X = np.sort(np.random.uniform(-2, 2, n)).reshape(-1,1)
y = np.sin(X.ravel()*np.pi) + np.random.normal(0, 0.3, n)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
X_plot = np.linspace(-2,2,200).reshape(-1,1)

lambdas = [0, 0.01, 1, 100]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, lam in zip(axes, lambdas):
    pipe = Pipeline([
        ('poly',  PolynomialFeatures(degree=10, include_bias=False)),
        ('scale', StandardScaler()),
        ('model', Ridge(alpha=lam) if lam > 0 else LinearRegression()),
    ])
    pipe.fit(X_tr, y_tr)
    tr_r2 = pipe.score(X_tr, y_tr)
    te_r2 = pipe.score(X_te, y_te)
    ax.scatter(X, y, s=15, alpha=0.5, color='steelblue')
    ax.plot(X_plot, pipe.predict(X_plot), 'r-', linewidth=2.5)
    label = f'lambda={lam}' if lam > 0 else 'OLS (lambda=0)'
    ax.set_title(f'{label}\nTrain R^2={tr_r2:.3f}\nTest  R^2={te_r2:.3f}', fontsize=9)
    ax.set_ylim(-2, 2); ax.grid(alpha=0.3)

plt.suptitle('Ridge: increasing lambda reduces overfitting (smoother curves)', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# Regularisation path: trace how coefficients shrink as lambda increases
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_std   = StandardScaler().fit_transform(housing.data)
y       = housing.target
names   = list(housing.feature_names)

lambdas = np.logspace(-3, 5, 200)
coefs   = np.array([Ridge(alpha=l).fit(X_std, y).coef_ for l in lambdas])

fig, ax = plt.subplots(figsize=(10, 5))
for j, name in enumerate(names):
    ax.semilogx(lambdas, coefs[:, j], linewidth=2, label=name)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('lambda (log scale)'); ax.set_ylabel('Coefficient value')
ax.set_title('Ridge regularisation path: coefficients shrink toward 0 but never reach it')
ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Find optimal lambda
from sklearn.linear_model import RidgeCV
rc = RidgeCV(alphas=lambdas, cv=5).fit(X_std, y)
print(f"Optimal lambda (5-fold RidgeCV): {rc.alpha_:.4f}")
print(f"Test R^2 at optimal lambda: {rc.score(X_std, y):.4f}")


## Key Takeaways

| | OLS | Ridge |
|--|-----|-------|
| Objective | Minimise SSR | Minimise SSR + lambda * sum(b^2) |
| Variable selection | No | No (keeps all, just smaller) |
| Multicollinearity | Unstable SEs | Stabilised |
| Tuning | None | Choose lambda via CV |
| Closed form | Yes | Yes: (XtX + lI)^-1 Xty |
| lambda=0 | OLS | OLS |

**Next: NB12 — Lasso (L1) which actually eliminates variables.**
